# Link chain

> A link-level transmission as a chain of stages, from the payload's bits to the bits decoded.

In [ ]:
#| default_exp radio.link.chain

In [ ]:
#| hide
from nbdev.showdoc import *

Link-level simulation in the sense of [Sionna PHY](https://nvlabs.github.io/sionna/phy/index.html): what happens
to the bits of each transmission. Every transmission goes through the same stages; only the choice at each stage
differs:

| stage | signals in → out | package | e.g. |
|---|---|---|---|
| source coding | payload → `bits` | `source` | `IndexCodec`, `UniformQuantizer`, `VectorQuantizer` |
| link adaptation | `sinr`, `num_bits` → `mcs`, `num_bits_per_symbol`, `coderate`, ... | `radio.link_adaptation` | `NRLinkAdaptation` |
| channel coding | `bits` → `codeword` | `coding` | `NoCoding`, `NRTBEncoder` |
| mapping | `codeword` → `x` | `modulation` | `Mapper(BPSK())`, `Mapper` of an MCS's QAM |
| waveform | `x`, `h`, `power` → `h_sym` | `waveform` | `SingleCarrier`, `OFDM` |
| the channel | `x`, `h_sym`, `no` → `y` | `channel` | `ApplyChannel` |
| equalization | `y`, `h_sym`, `no` → `x_hat`, `no_eff` | `equalization` | `OneTapEqualizer` |
| demapping | `x_hat`, `no_eff` → `llr` | `modulation` | `Demapper` |
| channel decoding | `llr` → `bits_hat` [, `crc_ok`] | `coding` | `HardDecision`, `NRTBDecoder` |
| source decoding | `bits_hat` → payload | `source` | the codec |

Each stage is a `Block` (see `block`) with named inputs and outputs, and the stages between the bits are a
`Pipeline`: a stage is swapped by giving another block with the same names (another code, a soft detector, an MMSE
equalizer, a Sionna block in `Named`).

**The signals**, one row per transmission of a group (`B`), as in Sionna:

* `bits` `(B, ..., k)` and `codeword` `(B, ..., n)`; `x` `(B, ..., S)`, the symbols.
* `h_sym`, the channel each symbol sees, `√P` included, broadcast against `x`; `no` `(B,)`, each transmission's
  noise power, the interference added; `y`, what the receiver gets.
* `x_hat`, `no_eff`: the equalized symbols and their noise; `llr`: soft bits, `log p(1) / p(0)` (Sionna's sign).
* `bits_hat`, and `crc_ok` when the code checks itself.
* A group's parameters, from link adaptation: `mcs`, `num_bits_per_symbol`, `coderate`, `tb_size`,
  `num_coded_bits`, `resource_elements`.

`LinkChain` runs the stages for a step's transmissions; `UncodedChain` and `NRChain` (see `radio.link.presets`) are
two of them, and `LinkLevelChannel` (see `radio.link.channel`) the radio channel whose receiver a chain is.

In [ ]:
#| export
from __future__ import annotations

import math
from collections import defaultdict
from typing import Sequence

import numpy as np

from comm_core.block import Block, Pipeline, _is_torch
from comm_core.channel.fading import effective_coefficient
from comm_core.modulation.constellation import Modulation
from comm_core.modulation.mapping import Mapper
from comm_core.source.codec import Codec, IndexCodec

## `LinkChain`

For each transmission of a step:

1. The payload becomes bits (`codec`).
2. **Shortcuts, the same for every chain:** an outage is not sent; nodes at the same place have a perfect link; a
   signal with no power, or drowned by an interferer at the receiver's place, is lost.
3. **Link adaptation** (`adaptation`, optional) gives each transmission its parameters, from its SINR.
4. **Groups.** The transmissions with the same bits' shape and parameters go through the stages together
   (`batch=True`), e.g. one call of Sionna's LDPC decoder for all of them; with `batch=False`, one at a time.
5. The bits that come out become the payload. Delivery is the CRC's (`crc_ok`) when the code has one. Without one,
   payloads arrive with their bit errors, or are dropped with `drop_corrupted` (an ideal CRC).

The stages share the chain's random generator, so its seed decides the noise. The results' `info` has
`bit_errors`, and the group parameters named in `report`.

In [ ]:
#| export
def _rows(x, rows):
    return x[rows] if isinstance(x, np.ndarray) else [x[k] for k in rows]


def _size(x) -> int:
    return int(x.numel()) if _is_torch(x) else int(np.size(x))


def _truncate(bits_hat, bits):
    "The decoded bits of one transmission, without the padding."
    bits_hat = np.asarray(bits_hat)
    if bits_hat.ndim == bits.ndim:                      # padded along the last axis (to whole symbols)
        return bits_hat[..., :bits.shape[-1]]
    return bits_hat.reshape(-1)[:bits.size].reshape(bits.shape)   # flattened into a transport block


class LinkChain(Block):
    "Link level: payload -> bits -> the stages (coding, mapping, the channel, equalization, demapping, decoding) -> payload."

    outputs = ('success', 'payload', 'error_rate', 'num_bits', 'num_symbols', 'info')

    def __init__(
        self,
        stages: Sequence[Block], # From `bits` to `bits_hat` (and `crc_ok` if the code checks itself)
        codec: Codec | None = None, # Payload <-> bits; None: `IndexCodec(256)`
        adaptation: Block | None = None, # Each transmission's parameters (e.g. its MCS), from `sinr` and `num_bits`
        batch: bool = True, # Transmissions with the same shape and parameters through the stages together
        drop_corrupted: bool = False, # Without a CRC: lose the packets with bit errors instead of delivering them
        report: Sequence[str] = (), # Parameters added to each result's info
    ):
        self.codec = codec if codec is not None else IndexCodec()
        self.stages = stages if isinstance(stages, Pipeline) else Pipeline(list(stages))
        self.adaptation, self.batch, self.drop_corrupted, self.report = adaptation, batch, drop_corrupted, tuple(report)
        self._share()

    def _share(self) -> None:
        "The stages draw from the chain's generator."
        for block in self.stages.walk():
            block.rng = self.rng

    def reset(self, seed: int | None = None) -> None:
        super().reset(seed)
        self._share()

    @property
    def needs_sinr(self) -> bool:
        "Whether the channel must give the SINR (for link adaptation)."
        return self.adaptation is not None and 'sinr' in self.adaptation.inputs

    @property
    def bits_per_symbol(self) -> int:
        "Of a fixed modulation (counts the symbols of what does not go through the stages); 1 if it varies."
        for block in self.stages:
            if isinstance(block, Mapper) and isinstance(block.modulation, Modulation):
                return block.modulation.bits_per_symbol
        return 1

    def call(
        self,
        transmission, # The transmissions (their payloads)
        h: np.ndarray, # h of each at each subcarrier, (n, K)
        power: np.ndarray, # Their transmit powers
        noise_power: float, # Noise power at the receivers (per symbol)
        interference: np.ndarray | None = None, # Power of the other transmissions at each receiver (added to the noise)
        active: np.ndarray | None = None, # Whether each one is sent (None: all); the others only count their bits
        group: Sequence | None = None, # The broadcast of each, None for a unicast (link adaptation: one MCS per broadcast)
        sinr: np.ndarray | None = None, # Their SINR at each subcarrier, for link adaptation
    ) -> tuple:
        "(success, payload, error_rate, num_bits, num_symbols, info) of each transmission."
        n = len(transmission)
        success, payload, error = np.zeros(n, dtype=bool), [None] * n, np.ones(n)
        num_bits, num_symbols, info = np.zeros(n, np.int64), np.zeros(n, np.int64), [{} for _ in range(n)]
        interference = np.zeros(n) if interference is None else np.asarray(interference, float)
        bits, todo = {}, []
        for k, t in enumerate(transmission):
            bits[k] = np.asarray(self.codec.encode(t.message.payload))
            num_bits[k], num_symbols[k] = bits[k].size, -(-int(bits[k].size) // self.bits_per_symbol)
            if active is not None and not active[k]:
                continue
            if np.isinf(h[k]).any():                     # same place: a perfect link
                success[k], payload[k], error[k], num_symbols[k], info[k] = True, t.message.payload, 0.0, 0, {'bit_errors': 0}
                continue
            noise = noise_power + float(interference[k])
            if not (float(power[k]) * abs(effective_coefficient(h[k])) ** 2 > 0 and math.isfinite(noise)):
                continue                                 # nothing to detect: no signal, or drowned (an interferer at the receiver's place)
            todo.append(k)
        if not todo:
            return success, payload, error, num_bits, num_symbols, info
        params = {}
        if self.adaptation is not None:
            params = self.adaptation(sinr=_rows(sinr, todo), num_bits=num_bits[todo],
                                     group=None if group is None else _rows(group, todo))
        groups = defaultdict(list)
        for i, k in enumerate(todo):
            key = (bits[k].shape, *(params[p][i].item() for p in params)) if self.batch else (k,)
            groups[key].append(i)
        for rows in groups.values():
            ks = [todo[i] for i in rows]
            state = dict(bits=np.stack([bits[k] for k in ks]), h=h[ks], power=np.asarray(power)[ks],
                         no=noise_power + interference[ks], **{p: v[rows[0]] for p, v in params.items()})
            out = self.stages(**{name: state[name] for name in self.stages.inputs if name in state})
            crc = out.get('crc_ok')
            for row, k in enumerate(ks):
                decoded = _truncate(out['bits_hat'][row], bits[k])
                errors = int((decoded != bits[k]).sum())
                success[k] = bool(crc[row]) if crc is not None else not (self.drop_corrupted and errors)
                payload[k] = self.codec.decode(decoded) if success[k] else None
                error[k], num_symbols[k] = errors / max(int(bits[k].size), 1), _size(out['x'][row])
                info[k] = {**{p: params[p][rows[row]].item() for p in self.report}, 'bit_errors': errors}
        return success, payload, error, num_bits, num_symbols, info

In [ ]:
show_doc(LinkChain.call)

### A chain of your own

Any stage can be replaced. Here a repetition code of rate 1/3 replaces `NoCoding` and `HardDecision`: each bit is sent
three times, and the decoder adds the three LLRs. Over the same channel, it makes fewer errors:

In [ ]:
from comm_core.channel.awgn import ApplyChannel
from comm_core.coding.uncoded import HardDecision, NoCoding
from comm_core.core import Message, Transmission
from comm_core.equalization.one_tap import OneTapEqualizer
from comm_core.modulation.constellation import BPSK
from comm_core.modulation.mapping import Demapper
from comm_core.radio.system.phy_abstraction import bpsk_bit_error_rate
from comm_core.waveform.single_carrier import SingleCarrier

In [ ]:
class Repeat(Block):
    "Each bit three times."
    outputs = ('codeword',)
    def call(self, bits):
        return np.repeat(bits, 3, axis=-1)

class Combine(Block):
    "The three LLRs of each bit added, then its sign."
    outputs = ('bits_hat',)
    def call(self, llr):
        return llr.reshape(*llr.shape[:-1], -1, 3).sum(-1) > 0

def chain(code, decode):
    mapper = Mapper(BPSK())
    return LinkChain([code, mapper, SingleCarrier(), ApplyChannel(), OneTapEqualizer(), Demapper(mapper), decode])

rows = [Transmission(Message('a', 'b', np.arange(200) % 256), 'a', 'b', 0.0) for _ in range(10)]
h, power = np.ones((10, 1), complex), np.ones(10)
uncoded, repeated = chain(NoCoding(), HardDecision()), chain(Repeat(), Combine())
uncoded.reset(seed=0); repeated.reset(seed=0)
ber = {name: np.mean(c(rows, h, power, noise_power=1.0)[2]) for name, c in (('uncoded', uncoded), ('repeated', repeated))}
assert abs(ber['uncoded'] - bpsk_bit_error_rate(1.0)) < 0.01 and ber['repeated'] < ber['uncoded'] / 2
print({k: round(v, 4) for k, v in ber.items()})

In [ ]:
# the shortcuts: not sent, the same place, no signal; the rest through the stages
success, payload, error, num_bits, num_symbols, info = uncoded(
    rows[:4], np.array([[1.0], [np.inf], [0.0], [1.0]], complex), np.ones(4), noise_power=1e-6, active=np.array([False, True, True, True]))
assert success.tolist() == [False, True, False, True] and num_bits.tolist() == [1600] * 4
assert num_symbols.tolist() == [1600, 0, 1600, 1600] and info[1] == {'bit_errors': 0} and info[3] == {'bit_errors': 0}

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()